In [ ]:
import sys
import os
import math
import time
import copy
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch

sys.path.insert(0, '.')

# Codebase imports
from src.config import DL_CONFIG, DIURNAL_WINDOW, DIURNAL_MIN_PERIODS, START_DATE
from src.data_main import load_and_clean_base_data
from src.features import make_har_features
from src.metrics import calculate_global_metrics
from src.data_helper import get_chunk_indices_strided

random.seed(42)

CONFIG = copy.deepcopy(DL_CONFIG)
CONFIG['data_path'] = 'all30min/'
CONFIG['num_lags'] = CONFIG['model']['context_len']

# Loading hparams for the codebase's data pipeline
LOADING_HPARAMS = {
    'exog_cols': 'none',
    'use_transform_exog': False,
    'use_diurnal': True,
    'allow_missing': False,
    'use_winsor': False,
}

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

In [ ]:
%%writefile backtest_worker.py
import torch
import torch.nn as nn
import math
import time
import gc
from datetime import datetime
from torch.func import vmap, functional_call, grad
from torch.amp import autocast
from transformers import PatchTSMixerConfig, PatchTSMixerModel, PreTrainedModel
from torch.optim.adamw import adamw


def log_to_file(message):
    timestamp = datetime.now().strftime('%H:%M:%S')
    with open('worker_log.txt', 'a') as f:
        f.write(f'{timestamp} - {message}\n')


# --- Log-Space Parameterized QLIKE Loss ---
def functional_qlike_loss(h_pred, target_sqrt):
    """
    QLIKE parameterized in log-space for numerical stability.

    h_pred: model output = log(sigma^2_pred), unconstrained real number
    target_sqrt: adj_RV (sqrt-space target from codebase pipeline)

    L = sigma^2_true * exp(-h_pred) + h_pred
    dL/dh = -sigma^2_true * exp(-h) + 1   (always bounded, no log(0) or div-by-zero)
    """
    target_sq = target_sqrt.squeeze().float() ** 2
    h = h_pred.squeeze().float()
    h = torch.clamp(h, min=-30.0, max=30.0)
    return target_sq * torch.exp(-h) + h


# --- MODEL ARCHITECTURE ---
class PatchTSMixerForecaster(PreTrainedModel):
    config_class = PatchTSMixerConfig

    def __init__(self, config):
        super().__init__(config)
        self.backbone = PatchTSMixerModel(config)

        dummy_input = torch.zeros(1, config.context_length, config.num_input_channels)
        with torch.no_grad():
            dummy_out = self.backbone(past_values=dummy_input).last_hidden_state

        self.num_patches = dummy_out.shape[2]
        self.flat_dim = self.num_patches * config.d_model

        self.head = nn.Linear(self.flat_dim, config.prediction_length)
        self.head.weight.data.normal_(0, 0.001)
        self.head.bias.data.fill_(0.0)

        self.post_init()

    def forward(self, past_values, future_values=None):
        outputs = self.backbone(past_values=past_values)
        last_hidden_state = outputs.last_hidden_state
        batch_size, num_channels, _, _ = last_hidden_state.shape
        flattened = last_hidden_state.view(batch_size, num_channels, -1)
        return self.head(flattened)


def get_model(cfg):
    config = PatchTSMixerConfig(
        context_length=cfg['context_len'],
        prediction_length=1,
        num_input_channels=cfg['num_input_channels'],
        d_model=cfg['hidden_dim'],
        num_layers=cfg.get('num_layers', 4),
        dropout=cfg['dropout'],
        patch_length=cfg['patch_len'],
        patch_stride=cfg['stride'],
        gated_attn=False,
        norm_type='layernorm',
        scaling=None
    )
    return PatchTSMixerForecaster(config)


# --- COMPILED TRAINING KERNEL ---
def make_train_kernel(base_model, param_keys, num_epochs, base_lr):

    def compute_loss_stateless(params, buffers, x, y):
        x_in = x.unsqueeze(-1)
        h_pred = functional_call(base_model, (params, buffers), args=(x_in,), kwargs={})
        return functional_qlike_loss(h_pred, y)

    batch_loss_fn = vmap(compute_loss_stateless, in_dims=(0, None, 0, 0), randomness='different')

    def train_loop(params, buffers, exp_avgs, exp_avg_sqs, step_tensors, X, y):

        for i in range(1, num_epochs + 1):
            def mean_loss(p):
                with autocast('cuda'):
                    losses = batch_loss_fn(p, buffers, X, y)
                    return losses.mean()

            grads_dict = grad(mean_loss)(params)

            grad_list = []
            found_inf = torch.tensor(False, device=X.device)

            for k in param_keys:
                g = grads_dict[k]
                g = torch.clamp(g, min=-5.0, max=5.0)
                grad_list.append(g)
                if not torch.isfinite(g).all():
                    found_inf = torch.tensor(True, device=X.device)

            if not found_inf:
                mutable_params = [params[k].clone() for k in param_keys]
                mutable_exp_avgs = [exp_avgs[k] for k in param_keys]
                mutable_exp_avg_sqs = [exp_avg_sqs[k] for k in param_keys]
                mutable_steps = [step_tensors[k] for k in param_keys]

                adamw(
                    params=mutable_params,
                    grads=grad_list,
                    exp_avgs=mutable_exp_avgs,
                    exp_avg_sqs=mutable_exp_avg_sqs,
                    max_exp_avg_sqs=[],
                    state_steps=mutable_steps,
                    amsgrad=False,
                    beta1=0.9,
                    beta2=0.999,
                    lr=base_lr,
                    weight_decay=0.01,
                    eps=1e-8,
                    maximize=False,
                    foreach=False,
                    capturable=True
                )
                params = {k: mutable_params[idx] for idx, k in enumerate(param_keys)}

        return params

    return train_loop


def gpu_worker(gpu_id, chunk_indices, model_config, train_config, shared_X, shared_y, shared_test_X, chunk_size, total_windows):
    try:
        device = torch.device(f'cuda:{gpu_id}')
        torch.cuda.set_device(device)
        torch.set_float32_matmul_precision('high')

        model_factory = lambda: get_model(model_config)

        base_model_train = model_factory().to(device)
        base_model_eval = model_factory().to(device)
        base_model_eval.eval()

        buffers = {n: b for n, b in base_model_train.named_buffers()}
        param_keys = [n for n, p in base_model_train.named_parameters()]

        steady_epochs = train_config['num_epochs']
        lr = train_config['learning_rate']

        log_to_file(f'Worker {gpu_id}: Compiling...')
        raw_steady = make_train_kernel(base_model_train, param_keys, steady_epochs, lr)
        steady_kernel = torch.compile(raw_steady, mode='default')

        # Prediction kernel: model outputs h = log(sigma^2)
        def stateless_fwd(p, b, x):
            x_input = x.unsqueeze(-1)
            h_pred = functional_call(base_model_eval, (p, b), args=(x_input,), kwargs={})
            return h_pred.squeeze(-1)

        predict_kernel = vmap(stateless_fwd, in_dims=(0, None, 0), out_dims=0)

        results = []
        param_shapes = {n: p.shape for n, p in base_model_train.named_parameters()}
        max_batch_size = chunk_size

        # Pre-allocate parameter tensors
        params_store = {}
        for name, shape in param_shapes.items():
            if len(shape) > 1:
                p = torch.empty((max_batch_size, *shape), device=device)
            else:
                p = torch.zeros((max_batch_size, *shape), device=device)
            params_store[name] = p.requires_grad_(True)

        for i, idx in enumerate(chunk_indices):
            start = idx * chunk_size
            end = min(start + chunk_size, total_windows)

            X_chunk = shared_X[start:end].to(device, non_blocking=True)
            y_chunk = shared_y[start:end].to(device, non_blocking=True)
            X_test_chunk = shared_test_X[start:end].to(device, non_blocking=True)

            curr_bs = X_chunk.shape[0]
            if curr_bs == 0:
                continue

            # --- INIT PARAMETERS ---
            for name, p_tensor in params_store.items():
                fan_in = p_tensor.shape[-1] if len(p_tensor.shape) > 2 else 1
                bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0.01

                if len(p_tensor.shape) > 1:
                    p_tensor.data.uniform_(-bound, bound)
                else:
                    p_tensor.data.zero_()

            current_params = params_store
            if curr_bs < max_batch_size:
                current_params = {k: v[:curr_bs] for k, v in params_store.items()}

            # --- INSTANCE NORMALIZATION (same concept as RollingRobustScaler) ---
            mean = X_chunk.mean(dim=2, keepdim=True)
            std = X_chunk.std(dim=2, keepdim=True) + 1e-8
            X_chunk = (X_chunk - mean) / std

            t_mean = X_test_chunk.mean(dim=2, keepdim=True)
            t_std = X_test_chunk.std(dim=2, keepdim=True) + 1e-8
            X_test_chunk = (X_test_chunk - t_mean) / t_std

            # --- TRAIN ---
            exp_avgs = {}
            exp_avg_sqs = {}
            step_tensors = {}

            for k, p in current_params.items():
                exp_avgs[k] = torch.zeros_like(p)
                exp_avg_sqs[k] = torch.zeros_like(p)
                step_tensors[k] = torch.tensor(0.0, device=device)

            final_params = steady_kernel(current_params, buffers, exp_avgs, exp_avg_sqs, step_tensors, X_chunk, y_chunk)

            detached_update = {
                k: v.detach().requires_grad_(True)
                for k, v in final_params.items()
            }

            # --- PREDICT ---
            with torch.no_grad():
                h_pred = predict_kernel(detached_update, buffers, X_test_chunk)
                # Convert from log-space to sqrt-space for Duan's smearing
                pred_sqrt = torch.exp(h_pred / 2.0)

            results.append({'chunk_index': idx, 'predictions': pred_sqrt.view(-1).cpu()})
            if i % 10 == 0:
                log_to_file(f'Worker {gpu_id}: Chunk {idx} done')

            del X_chunk, y_chunk, X_test_chunk, mean, std
            del final_params, detached_update, exp_avgs, exp_avg_sqs
            del h_pred, pred_sqrt

        return results

    except Exception as e:
        import traceback
        log_to_file(f'Worker {gpu_id} CRASHED:\n{traceback.format_exc()}')
        raise e

In [ ]:
def load_and_process_data(config, loading_hparams):
    """
    Load data using codebase pipeline: parquet -> clean -> diurnal+sqrt -> lags.
    Mirrors src/data.load_and_prep_data_strided() with dense consecutive lags.
    """
    print('Loading and cleaning data via codebase pipeline...')
    data, cols_to_transform = load_and_clean_base_data(loading_hparams, config['data_path'])

    target_col = 'adj_RV'
    consecutive_lags = list(range(1, config['model']['context_len'] + 1))

    print(f'Generating {len(consecutive_lags)} consecutive lag features...')
    feature_dict, feature_names = make_har_features(
        data, [target_col], consecutive_lags, 'raw', target_col
    )

    feature_df = pd.DataFrame(feature_dict, index=data.index)
    data = pd.concat([data, feature_df], axis=1)
    data = data.dropna()

    # Extract matrices (same pattern as load_and_prep_data_strided)
    X_np = data[feature_names].values.astype(np.float64)
    y_np = data[target_col].values.astype(np.float64)

    print(f'Data ready: {X_np.shape[0]} samples, {X_np.shape[1]} features')
    return X_np, y_np, data['t'], data['baseline_RV'].values, feature_names

In [ ]:
import backtest_worker
import importlib
import torch.multiprocessing as mp

importlib.reload(backtest_worker)


def run_multigpu_backtest(X_np, y_np, dates, baselines, config):
    """
    GPU-parallel backtest using get_chunk_indices_strided for chunk assignment
    and Duan's smearing (from save_chunk_results pattern) for reconstruction.
    """
    print(f'Starting Backtest on {config["gpu_count"]} GPUs')

    train_window = config['train_window']
    context_len = config['model']['context_len']
    stride_step = config['model']['prediction_length']
    chunk_size = config['train']['batch_size']

    # Use codebase's get_chunk_indices_strided to determine test indices
    # We create a dummy array to pass the shape
    total_samples = X_np.shape[0]
    total_test_indices = np.arange(train_window, total_samples)
    num_windows = len(total_test_indices)

    samples_per_window = train_window // context_len

    # Convert to tensors for GPU
    X_tensor = torch.tensor(X_np, dtype=torch.float32).share_memory_().pin_memory()
    y_tensor = torch.tensor(y_np, dtype=torch.float32).share_memory_().pin_memory()

    print(f'Windows: {num_windows} | Samples/Window: {samples_per_window} | Context: {context_len}')

    # --- Create Training Windows (3D) ---
    window_shape_X = (num_windows, samples_per_window, context_len)
    strides_X = (
        X_tensor.stride(0) * stride_step,
        X_tensor.stride(0) * context_len,
        X_tensor.stride(0)
    )
    all_train_X = torch.as_strided(X_tensor, size=window_shape_X, stride=strides_X)

    # --- Align Targets ---
    y_offset = y_tensor[context_len:]
    window_shape_y = (num_windows, samples_per_window, 1)
    strides_y = (
        y_offset.stride(0) * stride_step,
        y_offset.stride(0) * context_len,
        y_offset.stride(1)
    )
    all_train_y = torch.as_strided(y_offset, size=window_shape_y, stride=strides_y)

    # --- Create Test Tensor (OOS) ---
    X_test_start = X_tensor[train_window - context_len:]
    window_shape_test = (num_windows, 1, context_len)
    strides_test = (
        X_test_start.stride(0) * stride_step,
        X_test_start.stride(0),
        X_test_start.stride(0)
    )
    all_test_X = torch.as_strided(X_test_start, size=window_shape_test, stride=strides_test)

    print(f'Train X Shape: {all_train_X.shape}')
    print(f'Test X Shape:  {all_test_X.shape}')

    # --- Distribute chunks across GPUs using get_chunk_indices_strided pattern ---
    num_chunks = math.ceil(num_windows / chunk_size)
    chunk_indices = list(range(num_chunks))
    chunks_per_gpu = [chunk_indices[i::config['gpu_count']] for i in range(config['gpu_count'])]

    ctx = mp.get_context('spawn')
    with ctx.Pool(processes=config['gpu_count']) as pool:
        args = []
        for gpu_id in range(config['gpu_count']):
            args.append((
                gpu_id, chunks_per_gpu[gpu_id], config['model'], config['train'],
                all_train_X, all_train_y, all_test_X, chunk_size, num_windows
            ))
        results_nested = pool.starmap(backtest_worker.gpu_worker, args)

    # --- Aggregation ---
    print('Workers finished. Aggregating results...')
    flat_results = [item for sublist in results_nested for item in sublist]
    flat_results.sort(key=lambda x: x['chunk_index'])

    preds_sqrt = torch.cat([r['predictions'] for r in flat_results]).numpy()

    # Shape correction
    if len(preds_sqrt) != num_windows:
        print(f'Warning: Prediction shape mismatch. Expected {num_windows}, got {len(preds_sqrt)}.')
        ratio = len(preds_sqrt) / num_windows
        if ratio.is_integer() and ratio > 1:
            ratio = int(ratio)
            preds_sqrt = preds_sqrt.reshape(num_windows, ratio)[:, 0]
        else:
            preds_sqrt = preds_sqrt[:num_windows]

    # --- Reconstruct using Duan's smearing (from save_chunk_results pattern) ---
    pred_dates = dates.iloc[train_window:train_window + num_windows].values if hasattr(dates, 'iloc') else dates[train_window:train_window + num_windows]
    y_true_adj = y_np[train_window:train_window + num_windows]
    baselines_subset = baselines[train_window:train_window + num_windows]

    smear = np.mean((y_true_adj - preds_sqrt) ** 2)
    pred_raw = (preds_sqrt ** 2 + smear) * baselines_subset
    true_raw = (y_true_adj ** 2) * baselines_subset

    # Results DataFrame follows codebase convention (same as save_chunk_results)
    print(f'Saving results ({len(y_true_adj)} rows)...')
    results_df = pd.DataFrame({
        'date': pred_dates,
        'true_adj': y_true_adj,
        'pred_adj': preds_sqrt,
        'true_raw': true_raw,
        'pred_raw': pred_raw,
    })
    results_df.to_csv(config['output_path'], index=False)
    return results_df

In [ ]:
if __name__ == '__main__':
    # 1. Load and process data (codebase pipeline)
    X_np, y_np, dates, baselines, features = load_and_process_data(CONFIG, LOADING_HPARAMS)

    # 2. Run GPU backtest
    results = run_multigpu_backtest(X_np, y_np, dates, baselines, CONFIG)

    # 3. Evaluate using codebase metrics
    metrics = calculate_global_metrics(results)
    print(f"\nFinal QLIKE (Out-of-Sample): {metrics.get('qlike', float('nan')):.4f}")
    print(f"MSE: {metrics.get('mse', float('nan')):.6f}")
    print(f"MAE: {metrics.get('mae', float('nan')):.6f}")

    # 4. Visualization
    plt.figure(figsize=(12, 6))
    subset = results.iloc[-500:]
    plt.plot(subset.index, subset['true_raw'], label='True RV', alpha=0.6, color='black')
    plt.plot(subset.index, subset['pred_raw'], label='Predicted RV', alpha=0.8, color='#1f77b4')
    plt.title('Volatility Forecast: Model vs Ground Truth')
    plt.ylabel('Realized Volatility')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
y_true = results['true_raw'].values
y_pred = results['pred_raw'].values

plt.figure(figsize=(10, 10))
plt.scatter(y_true, y_pred, alpha=0.3, s=10, label='Predictions')

# Reference line
plt.plot([1e-6, 1e-2], [1e-6, 1e-2], 'k-', alpha=0.5, label='Perfect Match')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('True Volatility')
plt.ylabel('Predicted Volatility')
plt.title(f"Diagnostic: QLIKE = {metrics.get('qlike', float('nan')):.4f}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()